In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
from data_functions import *

In [2]:
PROJECT_ROOT = Path.cwd().parent

PATHS = {
    # Метки
    "train_labels": PROJECT_ROOT / "dataset" / "labels" / "labels_train.csv",
    "test_labels": PROJECT_ROOT / "dataset" / "labels" / "labels_test.csv",
    
    # Обучающая и тестовая телематика/расписание
    "train_traffic": PROJECT_ROOT / "dataset" / "train" / "traffic.csv",
    "train_schedule": PROJECT_ROOT / "dataset" / "train" / "schedule.csv",
    "test_traffic": PROJECT_ROOT / "dataset" / "test" / "traffic.csv",
    "test_schedule": PROJECT_ROOT / "dataset" / "test" / "schedule.csv",
    
    # Валидация (финальный инференс)
    "val_points": PROJECT_ROOT / "dataset" / "validate" / "points.csv",
    "val_traffic": PROJECT_ROOT / "dataset" / "validate" / "traffic.csv",
    "val_schedule_plan": PROJECT_ROOT / "dataset" / "validate" / "schedule_plan.csv",
    
    # Сабмит
    "sample_submission": PROJECT_ROOT / "dataset" / "sample_submission.csv",
    "output_submission": PROJECT_ROOT / "submission.csv",
}

PATHS

{'train_labels': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/labels/labels_train.csv'),
 'test_labels': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/labels/labels_test.csv'),
 'train_traffic': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/train/traffic.csv'),
 'train_schedule': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/train/schedule.csv'),
 'test_traffic': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/test/traffic.csv'),
 'test_schedule': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/test/schedule.csv'),
 'val_points': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/validate/points.csv'),
 'val_traffic': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/validate/traffic.csv'),
 'val_schedule_plan': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/validate/schedule_plan.csv'),
 'sample_submission': WindowsPath('c:/Users/kalin/Desktop/mos-transport/dataset/sample_submission.csv'),
 

In [3]:
# 1. Загрузка и предобработка обучающих данных
train_traffic = load_and_preprocess_traffic(PATHS["train_traffic"])
train_labels = load_and_preprocess_labels(PATHS["train_labels"])
train_schedule = load_and_preprocess_schedule(PATHS["train_schedule"])

# 2. Считаем статистики расписания по train
schedule_features = compute_schedule_features(train_schedule)

# 3. Собираем обучающий датасет
train_data_full = build_dataset(train_labels, train_traffic, schedule_features)

# 4. Загружаем и собираем тестовый набор для реальной валидации
test_traffic = load_and_preprocess_traffic(PATHS["test_traffic"])
test_labels = load_and_preprocess_labels(PATHS["test_labels"])
test_data_full = build_dataset(test_labels, test_traffic, schedule_features)

# 5. Обучаем модель с валидацией на labels_test.csv
model = train(train_df=train_data_full, val_df=test_data_full)

# 6. Загружаем и предобрабатываем набор периода validate (для финального сабмита)
val_points = load_and_preprocess_labels(PATHS["val_points"])
val_traffic = load_and_preprocess_traffic(PATHS["val_traffic"])
val_data_full = build_dataset(val_points, val_traffic, schedule_features)

# 7. Формируем финальный submission.csv с разделителем ';'
make_submission(
    model, val_data_full, PATHS["sample_submission"], PATHS["output_submission"]
)

0:	learn: 91.8791261	test: 92.5686958	best: 92.5686958 (0)	total: 153ms	remaining: 2m 32s
100:	learn: 69.7194660	test: 77.5180392	best: 77.5180392 (100)	total: 11.5s	remaining: 1m 42s
200:	learn: 63.7723807	test: 73.9571822	best: 73.9571822 (200)	total: 17.1s	remaining: 1m 8s
300:	learn: 58.6809167	test: 71.9750596	best: 71.9750596 (300)	total: 23.3s	remaining: 54.1s
400:	learn: 55.7115622	test: 71.3949818	best: 71.3823883 (397)	total: 29.6s	remaining: 44.2s
500:	learn: 53.5577445	test: 70.9194204	best: 70.9194204 (500)	total: 35.6s	remaining: 35.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 70.75230276
bestIteration = 529

Shrink model to first 530 iterations.
Модель сохранена в: c:\Users\kalin\Desktop\mos-transport\models\catboost_model.cbm

--- Результаты валидации ---
MAE Baseline (cur_dev_s): 93.36 s
MAE Model:               70.75 s
Улучшение:               22.61 s

Сабмит успешно сохранен в c:\Users\kalin\Desktop\mos-transport\submission.csv.
Колонки: ['sam